# 4x4 J1-J2 Graph Demo

This notebook shows how to use the current `nqs.graph` API to construct the geometry for a 2D J1-J2 model on a `4x4` square lattice with periodic boundary conditions.

At this stage the graph object does not store a Hamiltonian. Its job is to know the site ordering and the neighbor structure. The high-level API then decides which neighbor shell gets which coupling and color.

## Current idea

- `n=1` means nearest neighbors, used for `J1`
- `n=2` means diagonal next-nearest neighbors on the square lattice, used for `J2`
- `iter_edges(color, n)` creates colored edges on demand from the geometry
- the same edge lists can later feed both exact-diagonalization and NQS Hamiltonian builders

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent.parent
sys.path.insert(0, str(repo_root / "Balint" / "src"))

from nqs import SquareLattice

In [2]:
graph = SquareLattice(4, 4, pbc=True, ordering="row_major")

j1_edges = list(graph.iter_edges("J1", n=1))
j2_edges = list(graph.iter_edges("J2", n=2))

print("shape:", graph.shape)
print("nodes:", graph.n_nodes)
print("J1 edge count:", len(j1_edges))
print("J2 edge count:", len(j2_edges))
print("site 0 nearest neighbors:", graph.get_neighbors(0, 1))
print("site 0 next-nearest neighbors:", graph.get_neighbors(0, 2))
print("first 5 J1 edges:", j1_edges[:5])
print("first 5 J2 edges:", j2_edges[:5])

shape: (4, 4)
nodes: 16
J1 edge count: 32
J2 edge count: 32
site 0 nearest neighbors: (1, 3, 4, 12)
site 0 next-nearest neighbors: (5, 7, 13, 15)
first 5 J1 edges: [Edge(i=0, j=1, color='J1'), Edge(i=0, j=3, color='J1'), Edge(i=0, j=4, color='J1'), Edge(i=0, j=12, color='J1'), Edge(i=1, j=2, color='J1')]
first 5 J2 edges: [Edge(i=0, j=5, color='J2'), Edge(i=0, j=7, color='J2'), Edge(i=0, j=13, color='J2'), Edge(i=0, j=15, color='J2'), Edge(i=1, j=4, color='J2')]


## How this connects to the Hamiltonian references in `Anas/`

The reference class in `Anas/Hamiltonian2D.py` expects explicit edge lists and then applies local spin rules over those edges.

That means the graph module can stay purely geometric. The Hamiltonian layer can ask for:

- `graph.iter_edges("J1", n=1)` for nearest-neighbor couplings
- `graph.iter_edges("J2", n=2)` for diagonal next-nearest-neighbor couplings

and then pass only the `(i, j)` pairs to the Hamiltonian builder.

In [ ]:
sys.path.insert(0, str(repo_root))

from Anas.Hamiltonian2D import SpinGraph

J1 = 1.0
J2 = 0.5
g = 0.7

ham = SpinGraph(graph.n_nodes)
ham.add_zz([(edge.i, edge.j) for edge in j1_edges], J1)
ham.add_zz([(edge.i, edge.j) for edge in j2_edges], J2)
ham.add_x_field(g)

H = ham.build()
print("Hamiltonian shape:", H.shape)
print("Number of nonzero entries:", H.nnz)

## Later direction

The long-term `nqs.operator` layer should replace the reference Hamiltonian construction above.

The intended flow is:

1. `graph` provides ordered neighbor shells
2. a Hamiltonian builder converts those shells into local terms
3. `operator` turns local terms into connected matrix elements `H_{sigma', sigma}`

So this notebook is mainly a geometry and interoperability demo, not the final Hamiltonian API.